# Phase 3 · Lesson 3 — Alignment

**Mastering Agentic AI Certification · Pre-read**

> A fine-tuned model can follow instructions — but *should it follow every instruction*? **Alignment** is the stage that shapes a model's behaviour to be **helpful, honest, and harmless**, matching human preferences and values. It is what makes a model safe and reliable enough to act as an autonomous agent.

## 1. What is alignment?

**Alignment** trains a model to prefer responses that humans actually *want* — not just responses that are statistically likely or that literally satisfy an instruction.

Fine-tuning teaches *a* correct answer per example. Alignment goes further: it teaches the model to **rank** behaviours — to know that response A is *better* than response B even when both are fluent. It optimises for **preferences**, not single ground-truth labels.

The goal is usually summarised as the **3 H's**:

| Principle | Meaning |
|---|---|
| **Helpful** | Actually solves the user's problem, follows intent |
| **Honest** | Truthful; admits uncertainty; doesn't fabricate |
| **Harmless** | Refuses unsafe requests; avoids harmful output |

## 2. Where alignment sits in the LLM lifecycle

```
  [1] PRE-TRAINING            -> base model: learns language + knowledge
        |
  [2] FINE-TUNING / SFT       -> instruction-following on curated examples
        |
  [3] ALIGNMENT (RLHF / DPO)  -> helpful, honest, harmless behaviour        <-- YOU ARE HERE
        |
  [4] AGENTIC USE             -> tools, memory, planning, multi-step action
```

Alignment is step **[3]** — the final behavioural-shaping stage before a model is deployed as the brain of an agent.

## 3. From labels to *preferences*

The big shift in alignment: training data is no longer *(prompt → one correct answer)*. Instead humans (or a model) compare **two candidate responses** and say which is better:

```
Prompt:  "How do I deal with a stressful day?"
  Response A: "Take a short walk, breathe, and prioritise one task."   <- preferred
  Response B: "Just quit your job."                                    <- rejected
```

So the "label" here is a **comparison / ranking**, not a single answer. This is much easier for humans to provide reliably (judging is easier than authoring) and captures nuances of *quality* that a single gold answer cannot.

## 4. The main alignment methods

| Method | Core idea | Notes |
|---|---|---|
| **RLHF** (RL from Human Feedback) | Train a **reward model** on human preferences, then use **RL (PPO)** to push the LLM toward high-reward responses | Powerful; complex; the original ChatGPT recipe |
| **DPO** (Direct Preference Optimization) | Optimise the preference data **directly** with a simple classification-style loss — **no separate reward model or RL loop** | Simpler & stable; now very popular |
| **Constitutional AI / RLAIF** | Use an **AI model + a written set of principles** to generate the preference labels instead of humans | Scales feedback; reduces human labelling cost |

### The RLHF pipeline
```
 1. SFT model (from Phase 2)
 2. Collect human preferences:  prompt -> {response A, response B}, human picks winner
 3. Train a REWARD MODEL  r(prompt, response) -> scalar score, fit to those preferences
 4. RL (PPO): update the LLM to maximise reward, with a KL penalty that keeps it
    close to the SFT model so it doesn't "drift" or game the reward
```

In [ ]:
# Toy illustration of preference-based alignment (the RLHF/DPO intuition).
# A tiny "reward model" scores candidate responses; alignment = prefer the
# highest-reward one. No real neural net -- just to make the idea concrete.

# Hand-crafted reward features standing in for "what humans preferred".
def reward(response):
    r = 0.0
    helpful_words  = {"step", "first", "try", "consider", "because", "example"}
    harmful_words  = {"hack", "weapon", "illegal", "harm"}
    honest_words   = {"unsure", "approximately", "might", "depends"}
    words = set(response.lower().split())
    r += 1.0 * len(words & helpful_words)   # reward helpfulness
    r += 0.5 * len(words & honest_words)    # reward honesty / calibration
    r -= 3.0 * len(words & harmful_words)   # strongly penalise harm
    r -= 0.1 * max(0, len(words) - 20)      # mild penalty for rambling
    return round(r, 2)

prompt = "How can I improve my focus while studying?"
candidates = [
    "First, try the pomodoro technique because short focused steps help.",
    "Just hack your brain with illegal stimulants.",
    "It depends; you might consider removing distractions, for example your phone.",
]

scored = sorted(((reward(c), c) for c in candidates), reverse=True)
print("Prompt:", prompt, "\n")
for s, c in scored:
    print(f"reward={s:>5}  | {c}")
print("\nAligned model would output the TOP-scored response:")
print(">>", scored[0][1])

In [ ]:
# DPO intuition: given a (preferred, rejected) pair, the update should INCREASE
# the model's relative preference for the chosen response over the rejected one.
preferred = "First, try the pomodoro technique because short focused steps help."
rejected  = "Just hack your brain with illegal stimulants."

margin = reward(preferred) - reward(rejected)
print(f"reward(preferred) = {reward(preferred)}")
print(f"reward(rejected)  = {reward(rejected)}")
print(f"preference margin = {margin}  (DPO/RLHF pushes this to be large & positive)")

## 5. How alignment contributes to the Transformer

Like fine-tuning, alignment **does not change the Transformer architecture** — it changes *which* behaviours the same weights produce. The mechanics:

- **The policy is still the Transformer.** In RLHF/DPO the model being optimised (the "policy") is the SFT Transformer. RL/preference gradients flow back through the **same self-attention and feed-forward layers**, nudging their weights toward preferred outputs.
- **The reward model is *also* a Transformer.** It is typically a copy of the base/SFT Transformer with the language head replaced by a small **scalar (reward) head** — so even the "judge" reuses the same architecture.
- **A KL penalty anchors it to the base.** Updates are constrained to stay close to the SFT model's distribution, so the Transformer keeps the language and knowledge from earlier stages instead of collapsing or "reward-hacking".
- **Only the weights move, not the wiring.** Attention spans, layer counts, and context window are unchanged — alignment shifts the *probabilities* the Transformer assigns, biasing it toward helpful/honest/harmless continuations.

> **In one line:** alignment runs the **same Transformer** but optimises it against a *preference/reward signal* (often produced by another Transformer), shaping behaviour while leaving the architecture — and the knowledge from pre-training — intact.

## 6. Why alignment matters for agentic AI

- **Safety & refusals:** an autonomous agent that takes actions must reliably decline harmful or out-of-scope requests.
- **Instruction adherence under pressure:** alignment makes the agent follow *intent*, not just literal wording — crucial when chaining many steps.
- **Honesty / calibration:** an aligned agent says "I'm not sure" or calls a tool instead of hallucinating, which is essential for trustworthy multi-step reasoning.
- **Predictable behaviour:** preference optimisation reduces erratic outputs, so downstream planning and tool-use loops are stable.
- Alignment is the **last guardrail** before the model is handed autonomy in the agentic layer **[4]**.

## 7. Key takeaways

1. **Alignment** shapes a fine-tuned model to be **helpful, honest, and harmless**, matching human **preferences**.
2. Its training signal is a **comparison/ranking** of responses, not a single gold label.
3. **RLHF** = reward model + RL (PPO + KL penalty); **DPO** optimises preferences directly without a separate reward model; **Constitutional AI/RLAIF** uses AI-generated feedback.
4. It runs the **same Transformer** — the reward model is also a Transformer with a scalar head — changing weights/behaviour, not architecture.
5. Alignment is the **final safety and behaviour layer** that makes a model fit to act as an autonomous **agent**.

---
*Next (Phase 4):* the **agentic layer** — tools, memory, planning, and multi-step action built on top of an aligned model.

In [ ]:
# Environment sanity check — confirms the notebook is running in the dev container.
import sys, platform
print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())
print("Lesson 3 (Alignment) notebook is running ✓")